In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os

from utils.tools import *

In [ ]:
root = os.getcwd()
method = "Camera Params"
date = "8_17_26"
atom = "Nd_I"
top_n_lines = 5

path = os.path.join(root, method)
path = os.path.join(path, date)
results_path = os.path.join(root, "Results")
results_path = os.path.join(results_path, date)

results = {}
spectral_max = -np.inf
plotted_concentrations = []

fig, ax = plt.subplots(1)

for file in os.listdir(path):
    file_path = os.path.join(path, file)

    wavelength, spectrum = read_spe(file_path)
    total_area = np.trapezoid(spectrum, wavelength)
    spectrum = spectrum / total_area

    if spectral_max < spectrum.max():
        spectral_max = spectrum.max()
    

    gate_width, gate_delay = file_to_camera_params(file)

    ax.plot(wavelength, spectrum)

    results[file] = {
        "wavelength": wavelength,
        "spectrum": spectrum,
        "gate_width": gate_width,
        "gate_delay": gate_delay
    }


lines_df = read_referenece_lines(atom, wavelength, top_n_lines = top_n_lines)

ax.vlines(lines_df["wavelength"], ymin = spectrum.min(), ymax = 1.1 * spectral_max, linestyles = "--", color = "red")

for wl in lines_df["wavelength"]:
    ax.text(wl, 1.15 * spectral_max, f"{round(wl, 2)}", rotation = "vertical", ha = "center")

ax.grid(True)
ax.legend()
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Counts (a.u.)")
ax.set_ylim(0, 1.5 * spectral_max)
plt.show()

In [ ]:
N_points = 21
N_fit = 128
fit_method = "voigt"
maxfev = 5000
r2_limit = 0.5
debug = False

fit_param_names = {
    "gaussian": ["A", "mu", "sigma", "a", "b"],
    "lorentzian": ["A", "mu", "gamma", "a", "b"],
    "voigt": ["A", "mu", "sigma", "gamma", "a", "b"]
}

for file in results.keys():
    print("=" * 50)
    print(f"Fitting: {file}")
    print("=" * 50)
    data = results[file]

    x_full = data["wavelength"]
    y_full = data["spectrum"]
    areas = []
    intensities = []
    r2s = []

    for _, line in lines_df.iterrows():
        lamb0 = line["wavelength"]
        intensity = line["intensity_numeric"]

        idx = np.argmin(np.abs(x_full - lamb0))
        start = idx - N_points // 2
        end = idx + N_points // 2 + 1
        x = x_full[start : end]
        y = y_full[start : end]

        p0 = initial_guess(x, y, mode = fit_method)
        bounds = set_bounds(x, y, p0, mode = fit_method)

        if fit_method == "gaussian":
                    fit_func = gaussian_fit
        
        elif fit_method == "lorentzian":
            fit_func = lorentzian_fit
        
        elif fit_method == "voigt":
            fit_func = voigt_fit

        try:
              params, pcov = curve_fit(fit_func, x, y, p0 = p0, bounds = bounds, maxfev = maxfev)
              x_fit = np.linspace(x.min(), x.max(), N_fit)
              y_fit = fit_func(x_fit, *params)

        except Exception as e:
              print(f"An error occured: {e}")

              fig, ax = plt.subplots(1)
              ax.plot(x, y, "o")
              ax.grid(True)
              ax.set_xlabel("Wavelength (nm)")
              ax.set_ylabel("Counts (a.u.)")
              ax.set_title(f"Failed Fit: {file}")
              plt.show()

              continue

        if debug:
            fig, ax = plt.subplots(1)
            ax.plot(x, y, "o", color = "black")
            ax.plot(x_fit, y_fit, color = "red")
            ax.grid(True)
            ax.set_xlabel("Wavelength (nm)")
            ax.set_ylabel("Counts (a.u.)")
            ax.set_title(f"Debug Plot: {file}")
            scan_wavelength = data["scan_wavelength"]
            plt.show()

        x_fit = np.linspace(x.min(), x.max(), N_points)
        y_fit = fit_func(x_fit, *params)
        r2 = compute_r2(y, y_fit)

        if r2 > r2_limit:
              areas.append(intensity * compute_area(params, mode = fit_method))
              intensities.append(intensity)
              r2s.append(r2 * intensity)

    areas = np.array(areas)
    intensities = np.array(intensities)
    r2s = np.array(r2s)

    area = np.sum(areas) / (np.sum(intensities) + 1e-5)
    R2 = np.sum(r2s) / (np.sum(intensities) + 1e-5)

    results[file]["area"] = area
    results[file]["R2"] = R2

    print(f"R^2: {round(r2, 3)}")
    
    for name, value in zip(fit_param_names[fit_method], params):
        print(f"{name}: {round(value, 3)}")
        results[file][name] = value
    
    print(f"area: {round(area, 3)}")
    print("=" * 50)

In [ ]:
gate_widths = np.array([results[file]["gate_width"] for file in results.keys()])
gate_delays = np.array([results[file]["gate_delay"] for file in results.keys()])
areas = np.array([results[file]["area"] for file in results.keys()])

fig, ax = plt.subplots(1)
scatter = ax.plot(gate_widths, gate_delays, "o", c = areas, cmap = "magma")
fig.colorbar(scatter, ax = ax, label = "Average Peak Area")

plt.show()
